# 03 — Feature Engineering
**Goal:** Create new features that improve model performance and enable segmentation.

**New Features:**
- `tenure_group` — bucket tenure into 4 lifecycle stages
- `engagement_score` — count of 'Yes' services (0-7 scale)
- `charges_per_month` — TotalCharges / tenure (inferred average)
- `recency`, `frequency`, `monetary` — RFM proxies
- RFM quintile scores (R/F/M 1-5) and segment labels

In [ ]:
import sys; sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.feature_engineering import apply_all_features

df = pd.read_csv('../data/processed/cleaned_data.csv')
print(f'Before: {df.shape}')
df = apply_all_features(df)
print(f'After:  {df.shape}')
df[['customerID', 'tenure', 'engagement_score', 'charges_per_month', 'RFM_Score', 'RFM_Segment']].head(8)

In [ ]:
# ── Tenure Group Distribution ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

tenure_churn = df.groupby('tenure_group')['Churn'].mean() * 100
tenure_churn.plot(kind='bar', ax=axes[0], color='#f43f5e', edgecolor='white', rot=30)
axes[0].set_title('Churn Rate by Tenure Group', fontweight='bold')
axes[0].set_ylabel('Churn Rate (%)')

# Engagement score vs churn
eng_churn = df.groupby('engagement_score')['Churn'].mean() * 100
eng_churn.plot(kind='bar', ax=axes[1], color='#6366f1', edgecolor='white')
axes[1].set_title('Churn Rate by Engagement Score', fontweight='bold')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_xlabel('Number of services used (0-7)')

plt.tight_layout()
plt.show()
print('💡 More services → lower churn. 0-12 month customers churn most!')

In [ ]:
# ── RFM Segment Distribution ──────────────────────────
rfm_counts = df['RFM_Segment'].value_counts()
rfm_churn = df.groupby('RFM_Segment')['Churn'].mean() * 100

print('RFM Segment Distribution:')
print(rfm_counts)
print('\nChurn Rate by RFM Segment:')
print(rfm_churn.sort_values(ascending=False).round(1))

In [ ]:
# ── Scatter: RFM Score vs MonthlyCharges ─────────────
plt.figure(figsize=(10, 5))
scatter = plt.scatter(
    df['RFM_Score'], df['MonthlyCharges'],
    c=df['Churn'], cmap='RdBu_r', alpha=0.4, s=20
)
plt.colorbar(scatter, label='Churn (1=Yes)')
plt.title('RFM Score vs Monthly Charges (color=Churn)', fontweight='bold')
plt.xlabel('RFM Score (3-15)')
plt.ylabel('Monthly Charges ($)')
plt.tight_layout()
plt.show()

In [ ]:
# ── Save enriched dataset ─────────────────────────────
df.to_csv('../data/processed/cleaned_features.csv', index=False)
print(f'✅ Saved feature-engineered data: {df.shape}')
print(f'New columns added: {list(set(df.columns) - set(pd.read_csv("../data/processed/cleaned_data.csv").columns))}')